# Quantum Data Prep

Builds quantum feature tracks from the Track B predictors: every active encoding strategy × result getter × target, run on the Aer simulator. Each combination is saved as a dataframe track, `data/processed/track_<TRACK>_<target>.parquet`, ready for `TRACK_CONFIG` in `02_evaluation_loop`.

Each month is encoded on its own, so a month's quantum features depend only on that month's (causally scaled) inputs and never change between folds. That is why these tracks can be precomputed once, like the QRC/CRC reservoir tracks from `scripts/build_quantum_tracks.py`.

This notebook only holds configuration. The logic lives in `src/features/quantum/`:

- `encoding_strategy/` — turns a row of features into circuit parameter values
- `result_getters/` — runs the circuits on a primitive (or exactly) and returns the features
- `engineer_features.py` — encoding strategy + result getter, one row in, one row out

## 1. Environment setup

In [1]:
import os
import sys
from pathlib import Path

COLAB = False  # set to True when running in Google Colab

if COLAB:
    REPO_URL  = "https://github.com/manuelarceaguirre/capstone-quantum.git"
    BRANCH    = "clean"
    REPO_ROOT = Path("/content/capstone-quantum")

    %cd /content
    !rm -rf capstone-quantum
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/capstone-quantum
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime
    sys.path.insert(0, str(REPO_ROOT))
else:
    # local: walk up from the working directory to the repo root;
    # src/ is importable via `pip install -e .` (see environment.yml)
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

print("Repo root:", REPO_ROOT)

Repo root: C:\dev\projects\capstone-quantum


In [2]:
import json
import time
import warnings

import numpy as np
import pandas as pd
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2, SamplerV2

from src.features.quantum import engineer_features
from src.features.quantum.encoding_strategy import heisenberg_encoding_strategy, su_encoding_strategy
from src.features.quantum.result_getters import (
    bitstring_probability_result_getter,
    pce_result_getter,
    single_bit_sampler_result_getter,
    single_qubit_pauli_result_getter,
)

warnings.filterwarnings("ignore")

## 2. Configuration

In [3]:
# --- Paths ---
DATA_DIR = REPO_ROOT / "data" / "processed"
INPUT    = "track_b_{target}.parquet"   # Track B predictor set per target

# --- What to run ---
TARGET_LIST      = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
ACTIVE_ENCODINGS = ["su", "heisenberg", "zz"]                    # any keys of ENCODING_REGISTRY
ACTIVE_GETTERS   = ["pce", "pauli", "single_bit", "bitstring"]   # any keys of GETTER_REGISTRY
MAX_ROWS         = None   # set to a small int for a quick smoke test (output is then NOT saved)
RESUME           = False  # True: skip combinations whose track file already exists

# --- Aer simulator ---
SEED                = 42
ESTIMATOR_PRECISION = 0.01   # target standard error of each expectation value (estimator getters)
SHOTS               = 4096   # shots per circuit evaluation (sampler getters)
NOISE_BACKEND       = None   # e.g. qiskit_ibm_runtime.fake_provider.FakeTorino() to add its noise model

# --- Encoding / getter settings ---
NUM_TROTTER_STEPS    = 6   # Heisenberg and ZZ encodings
PCE_FEATURES         = 9   # number of Pauli correlators (pce getter), as in build_quantum_tracks.py
PCE_OPERATORS        = 2   # Paulis per correlator
MAX_BITSTRING_QUBITS = 8   # bitstring getter outputs 2^n columns; wider circuits are skipped
VALIDATE_ROWS        = 25  # rows re-run exactly (no shot noise) to check the Aer results

# --- Encoding registry: name -> factory(n_features) ---
# Inputs are min-max scaled causally below, so every encoding uses scale=False.
ENCODING_REGISTRY = {
    "su":         lambda n: su_encoding_strategy(n_qubits=int(np.ceil(n / 2)), reps=1, su2_gates=["ry", "rx"],
                                                 entanglement="full", scale=False),
    "heisenberg": lambda n: heisenberg_encoding_strategy(n_features=n, num_trotter_steps=NUM_TROTTER_STEPS,
                                                         seed=SEED, scale=False),
    "zz":         lambda n: heisenberg_encoding_strategy(n_features=n, num_trotter_steps=NUM_TROTTER_STEPS,
                                                         paulis=("ZZ",), seed=SEED, scale=False),
}

# --- Result getter registry: name -> (primitive kind, factory(n_qubits, primitive)) ---
# primitive=None gives exact simulation (used for validation).
GETTER_REGISTRY = {
    "pce":        ("estimator", lambda n, p: pce_result_getter(n, PCE_FEATURES, PCE_OPERATORS, estimator=p)),
    "pauli":      ("estimator", lambda n, p: single_qubit_pauli_result_getter(n, estimator=p)),
    "single_bit": ("sampler",   lambda n, p: single_bit_sampler_result_getter(n, sampler=p, shots=SHOTS)),
    "bitstring":  ("sampler",   lambda n, p: bitstring_probability_result_getter(n, sampler=p, shots=SHOTS)),
}


def track_name(encoding, getter):
    """Track key used in TRACK_CONFIG and in the file name."""
    return f"Q_{encoding}_{getter}"


print(f"Targets:   {TARGET_LIST}")
print(f"Encodings: {ACTIVE_ENCODINGS}")
print(f"Getters:   {ACTIVE_GETTERS}")
print(f"Aer:       precision {ESTIMATOR_PRECISION}, {SHOTS} shots, noise {'off' if NOISE_BACKEND is None else NOISE_BACKEND.name}")

Targets:   ['INDPRO', 'PAYEMS', 'CPIAUCSL', 'S&P 500']
Encodings: ['su', 'heisenberg', 'zz']
Getters:   ['pce', 'pauli', 'single_bit', 'bitstring']
Aer:       precision 0.01, 4096 shots, noise off


## 3. Load and scale inputs

Track B inputs are min-max scaled with expanding (causal) statistics, the same way `scripts/build_quantum_tracks.py` scales them: each month uses only the minimum and maximum seen up to that month. The encodings then map [0, 1] to rotation angles in [0, 2π].

In [4]:
def causal_minmax(df: pd.DataFrame) -> pd.DataFrame:
    """Scale each column to [0, 1] using min/max over rows up to and including t."""
    lo, hi = df.expanding().min(), df.expanding().max()
    return ((df - lo) / (hi - lo).replace(0, np.nan)).fillna(0.5).clip(0, 1)


inputs = {}
for target in TARGET_LIST:
    raw = pd.read_parquet(DATA_DIR / INPUT.format(target=target)).sort_index()
    inputs[target] = causal_minmax(raw).iloc[:MAX_ROWS]
    print(f"  {target:<10} {inputs[target].shape[1]:>2} features x {len(inputs[target])} months  "
          f"{inputs[target].index.min():%Y-%m} -> {inputs[target].index.max():%Y-%m}")

  INDPRO     10 features x 767 months  1962-04 -> 2026-02
  PAYEMS     10 features x 767 months  1962-04 -> 2026-02
  CPIAUCSL   12 features x 767 months  1962-04 -> 2026-02
  S&P 500    12 features x 767 months  1962-04 -> 2026-02


## 4. Aer primitives

Both primitives run locally on the Aer simulator through `qiskit_ibm_runtime`'s local mode, the same interface as IBM hardware. The result getters transpile each distinct circuit once and submit one primitive pub per circuit carrying every month's parameter values, so a whole track is one `run()` call.

In [5]:
simulator = (AerSimulator(seed_simulator=SEED) if NOISE_BACKEND is None
             else AerSimulator.from_backend(NOISE_BACKEND, seed_simulator=SEED))
PRIMITIVES = {
    "estimator": EstimatorV2(mode=simulator, options={"default_precision": ESTIMATOR_PRECISION}),
    "sampler":   SamplerV2(mode=simulator, options={"default_shots": SHOTS}),
}
print(simulator)

AerSimulator('aer_simulator')


## 5. Build the tracks

Loops encodings × getters × targets, saving each track file as soon as it is built, so an interrupted run keeps what it finished (rerun with `RESUME = True`).

In [6]:
combos = [(e, g, t) for e in ACTIVE_ENCODINGS for g in ACTIVE_GETTERS for t in TARGET_LIST]
log, built = [], {}
t_start = time.time()

for k, (enc_name, getter_name, target) in enumerate(combos, 1):
    track = track_name(enc_name, getter_name)
    path = DATA_DIR / f"track_{track}_{target}.parquet"
    data = inputs[target]
    encoding = ENCODING_REGISTRY[enc_name](data.shape[1])
    n_qubits = encoding.n_qubits
    kind, make_getter = GETTER_REGISTRY[getter_name]
    entry = {"track": track, "encoding": enc_name, "getter": getter_name, "target": target, "qubits": n_qubits}

    if getter_name == "bitstring" and n_qubits > MAX_BITSTRING_QUBITS:
        log.append({**entry, "status": f"skipped: {2**n_qubits} columns > 2^{MAX_BITSTRING_QUBITS}"})
        continue
    if RESUME and MAX_ROWS is None and path.exists():
        log.append({**entry, "status": "exists (resume)", "features": pd.read_parquet(path).shape[1]})
        continue

    t0 = time.time()
    features = engineer_features(data, encoding, make_getter(n_qubits, PRIMITIVES[kind]))
    features.index = data.index
    features.columns = [f"{track}_{c}" for c in features.columns]
    seconds = time.time() - t0

    built[(track, target)] = features
    if MAX_ROWS is None:
        features.to_parquet(path)
    log.append({**entry, "status": "built" if MAX_ROWS is None else "built (smoke test, not saved)",
                "features": features.shape[1], "seconds": round(seconds, 1)})

    elapsed = time.time() - t_start
    print(f"  [{k}/{len(combos)}] {track:<22} {target:<10} {n_qubits:>2} qubits  "
          f"{features.shape[1]:>3} features  {seconds:6.1f}s  · {elapsed/60:.1f} min elapsed")

log = pd.DataFrame(log)
print(f"Done in {(time.time() - t_start)/60:.1f} min")
log

  [23/48] Q_heisenberg_pauli     CPIAUCSL   13 qubits   39 features   166.1s  · 2.8 min elapsed


  [24/48] Q_heisenberg_pauli     S&P 500    13 qubits   39 features   167.4s  · 5.6 min elapsed


  [25/48] Q_heisenberg_single_bit INDPRO     11 qubits   11 features    16.5s  · 5.8 min elapsed


  [26/48] Q_heisenberg_single_bit PAYEMS     11 qubits   11 features    16.1s  · 6.1 min elapsed


  [27/48] Q_heisenberg_single_bit CPIAUCSL   13 qubits   13 features    20.8s  · 6.5 min elapsed


...

  [41/48] Q_zz_single_bit        INDPRO     11 qubits   11 features    12.6s  · 17.1 min elapsed


  [42/48] Q_zz_single_bit        PAYEMS     11 qubits   11 features    12.6s  · 17.3 min elapsed


  [43/48] Q_zz_single_bit        CPIAUCSL   13 qubits   13 features    14.7s  · 17.5 min elapsed


  [44/48] Q_zz_single_bit        S&P 500    13 qubits   13 features    14.6s  · 17.8 min elapsed
Done in 17.8 min


,track,encoding,getter,target,qubits,status,features,seconds
0,Q_su_pce,su,pce,INDPRO,5,exists (resume),9.0,NaN
1,Q_su_pce,su,pce,PAYEMS,5,exists (resume),9.0,NaN
2,Q_su_pce,su,pce,CPIAUCSL,6,exists (resume),9.0,NaN
3,Q_su_pce,su,pce,S&P 500,6,exists (resume),9.0,NaN
4,Q_su_pauli,su,pauli,INDPRO,5,exists (resume),15.0,NaN
5,Q_su_pauli,su,pauli,PAYEMS,5,exists (resume),15.0,NaN
6,Q_su_pauli,su,pauli,CPIAUCSL,6,exists (resume),18.0,NaN
7,Q_su_pauli,su,pauli,S&P 500,6,exists (resume),18.0,NaN
8,Q_su_single_bit,su,single_bit,INDPRO,5,exists (resume),5.0,NaN
9,Q_su_single_bit,su,single_bit,PAYEMS,5,exists (resume),5.0,NaN


## 6. Validate against exact simulation

Re-runs the first `VALIDATE_ROWS` months of every track exactly (no shot noise) and reports the largest difference from the Aer features. Expect differences of a few times `ESTIMATOR_PRECISION` for the estimator getters and about 1/√`SHOTS` for the sampler getters; much larger means something is wrong. With a noise model the gap also includes hardware noise.

In [7]:
rows = []
for (track, target), features in built.items():
    enc_name, getter_name = log.loc[(log["track"] == track) & (log["target"] == target), ["encoding", "getter"]].iloc[0]
    data = inputs[target].iloc[:VALIDATE_ROWS]
    encoding = ENCODING_REGISTRY[enc_name](data.shape[1])
    exact = engineer_features(data, encoding, GETTER_REGISTRY[getter_name][1](encoding.n_qubits, None)).to_numpy()
    diff = np.abs(features.iloc[:VALIDATE_ROWS].to_numpy() - exact)
    rows.append({"track": track, "target": target, "max |aer - exact|": diff.max(), "mean |aer - exact|": diff.mean()})

validation = pd.DataFrame(rows).set_index(["track", "target"]).unstack("target")
validation.round(4)

max |aer - exact|                          \
target                           CPIAUCSL  INDPRO  PAYEMS S&P 500   
track                                                               
Q_heisenberg_pauli                 0.0335     NaN     NaN  0.0295   
Q_heisenberg_single_bit            0.0241  0.0357  0.0192  0.0240   
Q_zz_pauli                         0.0364  0.0331  0.0304  0.0364   
Q_zz_pce                           0.0285  0.0295  0.0326  0.0242   
Q_zz_single_bit                    0.0230  0.0234  0.0234  0.0230   

                        mean |aer - exact|                          
target                            CPIAUCSL  INDPRO  PAYEMS S&P 500  
track                                                               
Q_heisenberg_pauli                  0.0070     NaN     NaN  0.0073  
Q_heisenberg_single_bit             0.0062  0.0059  0.0057  0.0060  
Q_zz_pauli                          0.0069  0.0067  0.0070  0.0071  
Q_zz_pce                            0.0082  0.0081  0.0075  0.0078  
Q_zz_single_bit                     0.0059  0.0055  0.0055  0.0059

## 7. Register the tracks

Writes `data/processed/quantum_tracks.json` (track names, settings and run log) and prints the `TRACK_CONFIG` entries to paste into `02_evaluation_loop`. Every new track counts as quantum in `03_plot_results`; add the names to its `QUANTUM_TRACKS`.

In [8]:
saved = log[log["status"].isin(["built", "exists (resume)"])]
tracks = sorted(saved["track"].unique())

if MAX_ROWS is None:
    manifest = {
        "built_at": pd.Timestamp.now(tz="UTC").isoformat(),
        "input": INPUT, "scaling": "causal min-max (expanding)",
        "settings": {"seed": SEED, "estimator_precision": ESTIMATOR_PRECISION, "shots": SHOTS,
                     "noise_backend": None if NOISE_BACKEND is None else NOISE_BACKEND.name,
                     "num_trotter_steps": NUM_TROTTER_STEPS, "pce_features": PCE_FEATURES,
                     "pce_operators": PCE_OPERATORS},
        "tracks": tracks,
        "log": json.loads(log.to_json(orient="records")),
    }
    (DATA_DIR / "quantum_tracks.json").write_text(json.dumps(manifest, indent=2))

print("# paste into TRACK_CONFIG in 02_evaluation_loop")
for track in tracks:
    print(f'    "{track}": {{t: {{"type": "dataframe", "path": DATA_DIR / f"track_{track}_{{t}}.parquet"}} for t in TARGET_LIST}},')
print(f"\n# 03_plot_results: QUANTUM_TRACKS = {['QRC', *tracks]}")

# paste into TRACK_CONFIG in 02_evaluation_loop
    "Q_heisenberg_pauli": {t: {"type": "dataframe", "path": DATA_DIR / f"track_Q_heisenberg_pauli_{t}.parquet"} for t in TARGET_LIST},
    "Q_heisenberg_pce": {t: {"type": "dataframe", "path": DATA_DIR / f"track_Q_heisenberg_pce_{t}.parquet"} for t in TARGET_LIST},
    "Q_heisenberg_single_bit": {t: {"type": "dataframe", "path": DATA_DIR / f"track_Q_heisenberg_single_bit_{t}.parquet"} for t in TARGET_LIST},
    "Q_su_bitstring": {t: {"type": "dataframe", "path": DATA_DIR / f"track_Q_su_bitstring_{t}.parquet"} for t in TARGET_LIST},
    "Q_su_pauli": {t: {"type": "dataframe", "path": DATA_DIR / f"track_Q_su_pauli_{t}.parquet"} for t in TARGET_LIST},
    "Q_su_pce": {t: {"type": "dataframe", "path": DATA_DIR / f"track_Q_su_pce_{t}.parquet"} for t in TARGET_LIST},
    "Q_su_single_bit": {t: {"type": "dataframe", "path": DATA_DIR / f"track_Q_su_single_bit_{t}.parquet"} for t in TARGET_LIST},
    "Q_zz_pauli": {t: {"type": "dataframe", "path":